# EXP-4: Fine-tune ConvNeXt-Tiny — Loss / Regularization

Notebook này thực hiện **EXP-4** trong pipeline fine-tuning ConvNeXt-Tiny:

```
BASELINE → EXP-1 → EXP-2 → EXP-3 → EXP-4 (Loss / Regularization) → EXP-5 → EXP-6 → FINAL
```

**Bối cảnh:** Qua BASELINE → EXP-1 → EXP-2 → EXP-3, `val_loss` liên tục tăng dần trong khi `val_accuracy`/`val_f1` gần như đứng yên (~0.85-0.87 accuracy, không tiến gần mục tiêu 95%). Đây là dấu hiệu **overconfidence/overfitting nhẹ**: model không học thêm được đặc trưng mới, chỉ ngày càng tự tin hơn với cả dự đoán đúng lẫn sai. Đổi scheduler ở EXP-3 chưa giải quyết được vì gốc rễ không nằm ở lịch giảm LR mà ở việc **thiếu cơ chế kìm hãm overconfidence**.

**3 thay đổi so với EXP-3 (giữ nguyên mọi thứ khác: discriminative LR, cosine scheduler, augmentation, batch size, epochs):**

1. **Label smoothing = 0.1** trong loss — công cụ trực tiếp nhất để trị overconfidence, không cho phép model đẩy xác suất về quá gần 0/1.
2. **Dropout (p=0.2)** thêm vào trước lớp Linear cuối của classifier — ConvNeXt-Tiny mặc định không có dropout ở head.
3. **Tăng weight_decay**: `1e-4 → 5e-4` — tăng regularization L2 tổng thể.

Vì thêm Dropout làm lệch index trong `classifier` (Linear chuyển từ `classifier[2]` sang `classifier[3]`), notebook có thêm logic **remap key checkpoint cũ** để load đúng trọng số từ EXP-1/2/3 (vốn lưu không có dropout).

**Nguồn checkpoint:** đọc `best_tracker.json` để lấy checkpoint tốt nhất tính đến hiện tại.

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm


In [ ]:
import os
import gc
import json
import time
import copy
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

print("All libraries imported successfully.")
print("PyTorch:", torch.__version__)


In [ ]:
# =========================
# Reproducibility
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =========================
# Device
# =========================

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


# =========================
# CUDA optimization
# =========================

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("CUDA optimization configured.")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
print(os.listdir("/content/drive/MyDrive")[:30])


In [ ]:
# =========================
# Google Drive
# =========================

DRIVE_ROOT = "/content/drive/MyDrive"

ZIP_NAME = "face_frames.zip"

ZIP_PATH = os.path.join(
    DRIVE_ROOT,
    ZIP_NAME
)


# =========================
# Local Colab
# =========================

EXTRACT_DIR = "/content/faceforensics"


# =========================
# Experiment output (dùng chung với BASELINE / EXP-1 / EXP-2 / EXP-3)
# =========================

EXPERIMENT_DIR = os.path.join(
    DRIVE_ROOT,
    "FaceForensics_Experiments"
)

MODEL_COMPARISON_CSV = os.path.join(
    EXPERIMENT_DIR,
    "results",
    "model_comparison.csv"
)


# =========================
# Thư mục pipeline fine-tune ConvNeXt-Tiny
# (dùng chung cho EXP-1 -> EXP-6 -> FINAL)
# =========================

FINETUNE_DIR = os.path.join(
    EXPERIMENT_DIR,
    "finetune_convnext_tiny"
)

FT_CHECKPOINT_DIR = os.path.join(
    FINETUNE_DIR,
    "checkpoints"
)

FT_HISTORY_DIR = os.path.join(
    FINETUNE_DIR,
    "histories"
)

FT_RESULT_DIR = os.path.join(
    FINETUNE_DIR,
    "results"
)

BEST_TRACKER_PATH = os.path.join(
    FINETUNE_DIR,
    "best_tracker.json"
)

EXPERIMENTS_LOG_PATH = os.path.join(
    FT_RESULT_DIR,
    "experiments_log.csv"
)


for folder in [
    EXPERIMENT_DIR,
    FINETUNE_DIR,
    FT_CHECKPOINT_DIR,
    FT_HISTORY_DIR,
    FT_RESULT_DIR
]:
    os.makedirs(folder, exist_ok=True)


print("ZIP:", ZIP_PATH)
print("Extract:", EXTRACT_DIR)
print("Finetune dir:", FINETUNE_DIR)
print("Best tracker:", BEST_TRACKER_PATH)


In [ ]:
# =========================
# Đọc best_tracker.json
# =========================

def init_best_tracker():
    """Fallback: nếu chưa từng chạy EXP nào, khởi tạo từ BASELINE (convnext_tiny)."""

    if os.path.exists(BEST_TRACKER_PATH):
        with open(BEST_TRACKER_PATH, "r") as f:
            return json.load(f)

    if not os.path.exists(MODEL_COMPARISON_CSV):
        raise FileNotFoundError(
            f"Không tìm thấy {MODEL_COMPARISON_CSV} và cũng chưa có "
            f"best_tracker.json. Hãy chạy Comparation_3_model.ipynb "
            f"và/hoặc các EXP trước đó."
        )

    comparison_df = pd.read_csv(MODEL_COMPARISON_CSV)

    baseline_row = comparison_df[
        comparison_df["model"] == "convnext_tiny"
    ].iloc[0]

    tracker = {
        "source_exp": "BASELINE",
        "checkpoint": baseline_row["checkpoint"],
        "best_val_f1": float(baseline_row["best_val_f1"]),
        "best_val_roc_auc": float(baseline_row["best_val_roc_auc"]),
        "best_val_accuracy": None,
    }

    with open(BEST_TRACKER_PATH, "w") as f:
        json.dump(tracker, f, indent=2)

    return tracker


best_tracker = init_best_tracker()

best_tracker.setdefault("best_val_accuracy", None)

CURRENT_BEST_CHECKPOINT = best_tracker["checkpoint"]

print("Best checkpoint hiện tại được lấy từ:", best_tracker["source_exp"])
print("Checkpoint path :", CURRENT_BEST_CHECKPOINT)
print("Best val_f1     :", best_tracker["best_val_f1"])
print("Best val_roc_auc:", best_tracker["best_val_roc_auc"])
print("Best val_accuracy:", best_tracker["best_val_accuracy"])

if not os.path.exists(CURRENT_BEST_CHECKPOINT):
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint: {CURRENT_BEST_CHECKPOINT}\n"
        "Hãy kiểm tra lại đường dẫn trên Google Drive."
    )


In [ ]:
if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / 1024**3
    print(f"Found ZIP: {ZIP_PATH}")
    print(f"ZIP size: {size_gb:.2f} GB")
else:
    raise FileNotFoundError(
        f"Không tìm thấy {ZIP_PATH}\n"
        "Hãy kiểm tra lại tên file hoặc vị trí trên Google Drive."
    )


In [ ]:
if os.path.exists(EXTRACT_DIR):
    print("Extract directory already exists.")
    print("Skip extraction.")

else:

    os.makedirs(EXTRACT_DIR, exist_ok=True)

    print("Extracting dataset...")
    print("This may take some time...")

    start_time = time.time()

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    elapsed = time.time() - start_time

    print(f"Extraction completed in {elapsed:.1f} seconds.")


In [ ]:
print(os.listdir(EXTRACT_DIR))


In [ ]:
def find_dataset_root(base_dir):

    candidates = []

    for root, dirs, files in os.walk(base_dir):

        dir_names = {d.lower() for d in dirs}

        if "train" in dir_names and "val" in dir_names and "test" in dir_names:
            candidates.append(root)

    if len(candidates) == 0:
        raise FileNotFoundError("Không tìm thấy thư mục chứa train/val/test.")

    return candidates[0]


DATA_DIR = find_dataset_root(EXTRACT_DIR)

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

print("DATA_DIR : ", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("TEST_DIR :", TEST_DIR)


In [ ]:
print("\nTrain:")
print(os.listdir(TRAIN_DIR))

print("\nValidation:")
print(os.listdir(VAL_DIR))


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def count_images(folder):

    count = 0

    for path in Path(folder).rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            count += 1

    return count


def count_class_images(split_dir):

    result = {}

    for class_dir in Path(split_dir).iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name.upper()
        result[class_name] = count_images(class_dir)

    return result


train_counts = count_class_images(TRAIN_DIR)
val_counts = count_class_images(VAL_DIR)

print("========== TRAIN ==========")
for cls, count in train_counts.items():
    print(f"{cls}: {count:,}")

print("\n========== VALIDATION ==========")
for cls, count in val_counts.items():
    print(f"{cls}: {count:,}")

print("\nTotal TRAIN:", f"{sum(train_counts.values()):,}")
print("Total VALIDATION:", f"{sum(val_counts.values()):,}")


In [ ]:
# =========================
# Image
# =========================

IMG_SIZE = 224


# =========================
# Training
# =========================
#
# EXP-4: "Loss / Regularization"
#   - Giu nguyen: 4 nhom discriminative LR, cosine scheduler (tu EXP-2/EXP-3).
#   - THAY DOI:
#       1) label_smoothing = 0.1 trong loss
#       2) them Dropout(p=0.2) truoc Linear cuoi cua classifier
#       3) tang weight_decay: 1e-4 -> 5e-4
#
# Muc tieu: giam overconfidence (val_loss tang lien tuc qua EXP-1..3
# trong khi accuracy dung yen ~0.85-0.87).

BATCH_SIZE = 64
EPOCHS = 10

WEIGHT_DECAY = 5e-4   # <-- tang tu 1e-4 (EXP-1,2,3)

LABEL_SMOOTHING = 0.1  # <-- MOI
DROPOUT_P = 0.2         # <-- MOI

# ---- Discriminative learning rates (giữ nguyên như EXP-2/EXP-3) ----
LR_STEM_STAGE1   = 1e-6   # features[0:2]  (stem + stage 1)
LR_STAGE2        = 3e-6   # features[2:4]  (downsample + stage 2)
LR_STAGE3        = 1e-5   # features[4:6]  (downsample + stage 3)
LR_STAGE4_HEAD   = 3e-5   # features[6:8] + classifier (downsample + stage4 + head)

LEARNING_RATE = LR_STAGE4_HEAD  # dùng để log/so sánh chung

# ---- Cosine Annealing scheduler (giữ nguyên như EXP-3) ----
COSINE_T_MAX = EPOCHS
COSINE_ETA_MIN_RATIO = 0.01


# =========================
# DataLoader
# =========================

NUM_WORKERS = 4
PIN_MEMORY = (device.type == "cuda")
PERSISTENT_WORKERS = (NUM_WORKERS > 0)
PREFETCH_FACTOR = 2


# =========================
# Mixed Precision
# =========================

USE_AMP = (device.type == "cuda")


# =========================
# Classes
# =========================

CLASS_TO_LABEL = {"REAL": 0, "FAKE": 1}
LABEL_TO_CLASS = {0: "REAL", 1: "FAKE"}


# =========================
# Experiment identity
# =========================

MODEL_NAME = "convnext_tiny"
EXP_NAME = "exp4_loss_regularization"
EXP_DESCRIPTION = (
    "Giu nguyen discriminative LR + cosine scheduler tu EXP-3. "
    "Them label_smoothing=0.1, Dropout(p=0.2) truoc Linear cuoi, "
    "tang weight_decay 1e-4 -> 5e-4 de giam overconfidence "
    "(val_loss tang lien tuc qua EXP-1..3)"
)


print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("EPOCHS:", EPOCHS)
print("WEIGHT_DECAY:", WEIGHT_DECAY, "(tang tu 1e-4)")
print("LABEL_SMOOTHING:", LABEL_SMOOTHING)
print("DROPOUT_P:", DROPOUT_P)
print("LR groups:")
print("  stem+stage1 :", LR_STEM_STAGE1)
print("  stage2      :", LR_STAGE2)
print("  stage3      :", LR_STAGE3)
print("  stage4+head :", LR_STAGE4_HEAD)
print("Cosine T_max:", COSINE_T_MAX)
print("AMP:", USE_AMP)
print("EXP_NAME:", EXP_NAME)


In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


print("Transforms ready (unchanged from EXP-3).")


In [ ]:
def create_dataframe(split_dir):

    records = []
    split_path = Path(split_dir)

    for class_dir in split_path.iterdir():

        if not class_dir.is_dir():
            continue

        class_name = class_dir.name.upper()

        if class_name not in CLASS_TO_LABEL:
            print(f"Warning: ignored class {class_name}")
            continue

        label = CLASS_TO_LABEL[class_name]

        for image_path in class_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({
                    "image_path": str(image_path),
                    "label": label,
                    "class_name": class_name
                })

    return pd.DataFrame(records)


train_df = create_dataframe(TRAIN_DIR)
val_df = create_dataframe(VAL_DIR)

print("TRAIN:", f"{len(train_df):,}")
print("VAL:", f"{len(val_df):,}")


In [ ]:
print("\nTRAIN")
print(train_df["class_name"].value_counts())

print("\nVAL")
print(val_df["class_name"].value_counts())


In [ ]:
class FaceForensicsDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]
        image_path = row["image_path"]
        label = int(row["label"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"\nError loading:\n{image_path}")
            raise e

        if self.transform is not None:
            image = self.transform(image)

        return image, label


In [ ]:
train_dataset = FaceForensicsDataset(train_df, transform=train_transform)
val_dataset = FaceForensicsDataset(val_df, transform=val_transform)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None
)


print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


In [ ]:
def create_model(model_name):
    """
    EXP-4: classifier có thêm Dropout(p=DROPOUT_P) trước Linear cuối.

    Cấu trúc classifier gốc (EXP-1,2,3):
        (0) LayerNorm2d
        (1) Flatten
        (2) Linear

    Cấu trúc classifier MỚI (EXP-4):
        (0) LayerNorm2d
        (1) Flatten
        (2) Dropout(p=DROPOUT_P)
        (3) Linear
    """

    if model_name == "convnext_tiny":

        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)

        in_features = model.classifier[-1].in_features

        model.classifier = nn.Sequential(
            model.classifier[0],          # LayerNorm2d
            model.classifier[1],          # Flatten
            nn.Dropout(p=DROPOUT_P),      # MOI: Dropout
            nn.Linear(in_features, 2)     # Linear cuoi
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model


def remap_legacy_classifier_keys(state_dict):
    """
    Checkpoint từ EXP-1/2/3 có classifier dạng (0,1,2) = (LayerNorm2d, Flatten, Linear).
    Model EXP-4 có classifier dạng (0,1,2,3) = (LayerNorm2d, Flatten, Dropout, Linear).
    Dropout không có tham số, nên chỉ cần đổi tên key "classifier.2.*" (Linear cũ)
    thành "classifier.3.*" (Linear mới) nếu phát hiện checkpoint cũ.
    """

    has_legacy_linear = (
        "classifier.2.weight" in state_dict
        and "classifier.3.weight" not in state_dict
    )

    if not has_legacy_linear:
        # Checkpoint đã ở format mới (ví dụ chạy lại chính EXP-4)
        return state_dict

    new_state_dict = {}

    for key, value in state_dict.items():

        if key.startswith("classifier.2."):
            new_key = key.replace("classifier.2.", "classifier.3.")
        else:
            new_key = key

        new_state_dict[new_key] = value

    print("  (Đã remap classifier.2.* -> classifier.3.* cho checkpoint cũ không có Dropout)")

    return new_state_dict


def load_finetune_weights(model, checkpoint_path):
    """Load trọng số từ best checkpoint trước đó, tự động remap nếu cần."""

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False  # checkpoint tu train, tin tuong duoc (PyTorch 2.6+ mac dinh weights_only=True)
    )

    state_dict = checkpoint.get("model_state_dict", checkpoint)

    state_dict = remap_legacy_classifier_keys(state_dict)

    model.load_state_dict(state_dict)

    print(f"Loaded weights from: {checkpoint_path}")

    if "metrics" in checkpoint:
        print("  Source metrics:", checkpoint["metrics"])

    return model


def unfreeze_all(model):

    for param in model.parameters():
        param.requires_grad = True

    return model


In [ ]:
model = create_model(MODEL_NAME)

model = load_finetune_weights(model, CURRENT_BEST_CHECKPOINT)

model = unfreeze_all(model)

model = model.to(device)

if device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)


total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(
    f"{MODEL_NAME:20s}"
    f" | Params: {total_params / 1e6:.2f}M"
    f" | Trainable: {trainable_params / 1e6:.2f}M"
)

assert total_params == trainable_params, "Vẫn còn tham số bị đóng băng!"
print("✓ Toàn bộ tham số đã unfreeze (trainable = total).")

print("\nClassifier structure:")
print(model.classifier)


In [ ]:
class_counts = train_df["label"].value_counts().sort_index()

print("Class counts:")
for label, count in class_counts.items():
    print(LABEL_TO_CLASS[label], ":", f"{count:,}")

total_samples = len(train_df)
num_classes = 2

class_weights = []
for label in range(num_classes):
    count = class_counts[label]
    weight = total_samples / (num_classes * count)
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print("\nClass weights:")
for label, weight in enumerate(class_weights):
    print(LABEL_TO_CLASS[label], ":", f"{weight.item():.4f}")


In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING   # <-- MOI: giam overconfidence
)

print(criterion)


In [ ]:
def build_discriminative_param_groups(model):
    """Giữ nguyên cách chia nhóm như EXP-2/EXP-3 (4 nhóm theo độ sâu)."""

    features = model.features

    group_stem_stage1 = list(features[0].parameters()) + list(features[1].parameters())
    group_stage2 = list(features[2].parameters()) + list(features[3].parameters())
    group_stage3 = list(features[4].parameters()) + list(features[5].parameters())

    group_stage4_head = (
        list(features[6].parameters())
        + list(features[7].parameters())
        + list(model.avgpool.parameters())
        + list(model.classifier.parameters())
    )

    n_grouped = sum(
        p.numel()
        for g in [group_stem_stage1, group_stage2, group_stage3, group_stage4_head]
        for p in g
    )
    n_total = sum(p.numel() for p in model.parameters())

    assert n_grouped == n_total, (
        f"Param groups không khớp tổng số tham số! "
        f"grouped={n_grouped:,} vs total={n_total:,}."
    )

    param_groups = [
        {"params": group_stem_stage1, "lr": LR_STEM_STAGE1, "name": "stem+stage1"},
        {"params": group_stage2,      "lr": LR_STAGE2,      "name": "stage2"},
        {"params": group_stage3,      "lr": LR_STAGE3,      "name": "stage3"},
        {"params": group_stage4_head, "lr": LR_STAGE4_HEAD, "name": "stage4+head"},
    ]

    print("Discriminative LR param groups:")
    for g in param_groups:
        n_params = sum(p.numel() for p in g["params"])
        print(f"  {g['name']:15s} | LR={g['lr']:.1e} | Params={n_params/1e6:.2f}M")

    return param_groups


def create_optimizer(model):

    param_groups = build_discriminative_param_groups(model)

    return torch.optim.AdamW(
        param_groups,
        weight_decay=WEIGHT_DECAY   # 5e-4 (tang tu 1e-4)
    )


In [ ]:
def create_scheduler(optimizer):
    """Giữ nguyên CosineAnnealingLR (custom lambda) như EXP-3."""

    import math

    def cosine_factor(epoch):
        progress = epoch / max(1, COSINE_T_MAX)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return COSINE_ETA_MIN_RATIO + (1 - COSINE_ETA_MIN_RATIO) * cosine

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=[cosine_factor for _ in optimizer.param_groups]
    )

    return scheduler


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):

    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Train", leave=False)

    for images, labels in progress:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if device.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    epoch_f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)

    return epoch_loss, epoch_f1


In [ ]:
def validate(model, loader, criterion):

    model.eval()   # Dropout tu dong tat trong eval mode

    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if device.type == "cuda":
                images = images.contiguous(memory_format=torch.channels_last)

            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probabilities = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probabilities[:, 1].cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
    recall = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)

    return {
        "loss": epoch_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc
    }


In [ ]:
def save_checkpoint(model, optimizer, scheduler, scaler, epoch, metrics, best_f1):

    checkpoint_path = os.path.join(
        FT_CHECKPOINT_DIR,
        f"{MODEL_NAME}_{EXP_NAME}_best.pth"
    )

    checkpoint = {

        "model_name": MODEL_NAME,
        "exp_name": EXP_NAME,
        "finetuned_from": CURRENT_BEST_CHECKPOINT,

        "epoch": epoch,

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if USE_AMP else None,

        "best_f1": best_f1,
        "metrics": metrics,

        "config": {
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "dropout_p": DROPOUT_P,
            "lr_stem_stage1": LR_STEM_STAGE1,
            "lr_stage2": LR_STAGE2,
            "lr_stage3": LR_STAGE3,
            "lr_stage4_head": LR_STAGE4_HEAD,
            "scheduler": "CosineAnnealingLR (custom lambda)",
            "cosine_t_max": COSINE_T_MAX,
            "cosine_eta_min_ratio": COSINE_ETA_MIN_RATIO,
            "seed": SEED
        }
    }

    torch.save(checkpoint, checkpoint_path)

    return checkpoint_path


In [ ]:
def train_model():

    print("\n")
    print("=" * 70)
    print(f"START FINE-TUNE: {MODEL_NAME} | {EXP_NAME}")
    print(f"Description: {EXP_DESCRIPTION}")
    print(f"Starting from: {CURRENT_BEST_CHECKPOINT}")
    print("=" * 70)

    optimizer = create_optimizer(model)
    scheduler = create_scheduler(optimizer)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    history = []
    best_f1 = -1.0
    best_auc = -1.0
    best_accuracy = -1.0
    best_loss_at_best = None
    best_epoch = -1

    total_start = time.time()

    for epoch in range(EPOCHS):

        epoch_start = time.time()

        print(f"\nEpoch {epoch + 1}/{EPOCHS}")

        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler
        )

        val_metrics = validate(model, val_loader, criterion)

        scheduler.step()

        current_lrs = [g["lr"] for g in optimizer.param_groups]

        epoch_time = time.time() - epoch_start

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_f1": train_f1,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "lr_stem_stage1": current_lrs[0],
            "lr_stage2": current_lrs[1],
            "lr_stage3": current_lrs[2],
            "lr_stage4_head": current_lrs[3],
            "epoch_time_sec": epoch_time
        }

        history.append(row)

        print(f"Train Loss : {train_loss:.4f}")
        print(f"Train F1   : {train_f1:.4f}")
        print(f"Val Loss   : {val_metrics['loss']:.4f}")
        print(f"Val Acc    : {val_metrics['accuracy']:.4f}")
        print(f"Val F1     : {val_metrics['f1']:.4f}")
        print(f"Val ROC-AUC: {val_metrics['roc_auc']:.4f}")
        print(f"LRs (stem->head): {[f'{lr:.1e}' for lr in current_lrs]}")
        print(f"Epoch time : {epoch_time:.1f}s")

        is_better = (
            val_metrics["f1"] > best_f1
            or (val_metrics["f1"] == best_f1 and val_metrics["roc_auc"] > best_auc)
        )

        if is_better:

            best_f1 = val_metrics["f1"]
            best_auc = val_metrics["roc_auc"]
            best_accuracy = val_metrics["accuracy"]
            best_loss_at_best = val_metrics["loss"]
            best_epoch = epoch + 1

            checkpoint_path = save_checkpoint(
                model, optimizer, scheduler, scaler, epoch + 1, val_metrics, best_f1
            )

            print(f"✓ BEST MODEL (EXP-4) SAVED")
            print(f"  F1: {best_f1:.4f}")
            print(f"  AUC: {best_auc:.4f}")
            print(f"  Accuracy: {best_accuracy:.4f}")
            print(f"  Val Loss tại best: {best_loss_at_best:.4f}")

    total_time = time.time() - total_start

    history_df = pd.DataFrame(history)

    history_path = os.path.join(
        FT_HISTORY_DIR,
        f"{MODEL_NAME}_{EXP_NAME}_history.csv"
    )

    history_df.to_csv(history_path, index=False)

    summary = {
        "exp_name": EXP_NAME,
        "description": EXP_DESCRIPTION,
        "model": MODEL_NAME,
        "finetuned_from": CURRENT_BEST_CHECKPOINT,
        "learning_rate": LEARNING_RATE,
        "best_epoch": best_epoch,
        "best_val_f1": best_f1,
        "best_val_roc_auc": best_auc,
        "best_val_accuracy": best_accuracy,
        "best_val_loss": best_loss_at_best,
        "total_training_time_sec": total_time,
        "average_epoch_time_sec": history_df["epoch_time_sec"].mean(),
        "checkpoint": os.path.join(
            FT_CHECKPOINT_DIR, f"{MODEL_NAME}_{EXP_NAME}_best.pth"
        ),
        "history": history_path
    }

    print("\n")
    print("=" * 70)
    print(f"FINISHED: {EXP_NAME}")
    print("=" * 70)
    print(f"Best Epoch   : {best_epoch}")
    print(f"Best F1      : {best_f1:.4f}")
    print(f"Best AUC     : {best_auc:.4f}")
    print(f"Best Accuracy: {best_accuracy:.4f}  (mục tiêu >= 0.95)")
    print(f"Val Loss tại best epoch: {best_loss_at_best:.4f}")
    print(f"Avg epoch    : {summary['average_epoch_time_sec']:.1f}s")
    print(f"Total time   : {total_time / 60:.2f} min")

    return summary, history_df


In [ ]:
exp4_summary, exp4_history = train_model()


In [ ]:
# =========================
# So sánh EXP-4 với best hiện tại, cập nhật best_tracker.json NẾU tốt hơn
# =========================

is_new_best = (
    exp4_summary["best_val_f1"] > best_tracker["best_val_f1"]
    or (
        exp4_summary["best_val_f1"] == best_tracker["best_val_f1"]
        and exp4_summary["best_val_roc_auc"] > best_tracker["best_val_roc_auc"]
    )
)

print(f"Previous best ({best_tracker['source_exp']}):")
print(f"  val_f1      = {best_tracker['best_val_f1']:.4f}")
print(f"  val_roc_auc = {best_tracker['best_val_roc_auc']:.4f}")
print(f"  val_accuracy= {best_tracker['best_val_accuracy']}")

print(f"\nEXP-4 result:")
print(f"  val_f1      = {exp4_summary['best_val_f1']:.4f}")
print(f"  val_roc_auc = {exp4_summary['best_val_roc_auc']:.4f}")
print(f"  val_accuracy= {exp4_summary['best_val_accuracy']:.4f}")
print(f"  val_loss    = {exp4_summary['best_val_loss']:.4f}")

if is_new_best:

    best_tracker = {
        "source_exp": EXP_NAME,
        "checkpoint": exp4_summary["checkpoint"],
        "best_val_f1": exp4_summary["best_val_f1"],
        "best_val_roc_auc": exp4_summary["best_val_roc_auc"],
        "best_val_accuracy": exp4_summary["best_val_accuracy"],
    }

    with open(BEST_TRACKER_PATH, "w") as f:
        json.dump(best_tracker, f, indent=2)

    print(
        f"\n✓ EXP-4 CẢI THIỆN kết quả. "
        f"best_tracker.json đã được cập nhật -> nguồn cho EXP-5 sẽ là EXP-4."
    )

else:

    print(
        f"\n✗ EXP-4 KHÔNG cải thiện so với best hiện tại "
        f"({best_tracker['source_exp']}). "
        f"best_tracker.json GIỮ NGUYÊN -> EXP-5 vẫn fine-tune từ "
        f"{best_tracker['source_exp']}, không phải từ EXP-4."
    )

# So sánh xu hướng val_loss cuối kỳ so với EXP-3 để xem regularization có hiệu quả không
exp3_history_path = os.path.join(
    FT_HISTORY_DIR, "convnext_tiny_exp3_cosine_scheduler_history.csv"
)

if os.path.exists(exp3_history_path):

    exp3_history_check = pd.read_csv(exp3_history_path)

    exp3_final_loss = exp3_history_check["val_loss"].iloc[-1]
    exp4_final_loss = exp4_history["val_loss"].iloc[-1]

    print(f"\nVal loss cuối kỳ EXP-3: {exp3_final_loss:.4f}")
    print(f"Val loss cuối kỳ EXP-4: {exp4_final_loss:.4f}")

    if exp4_final_loss < exp3_final_loss:
        print("✓ Regularization (label smoothing + dropout + weight decay) đã giúp giảm val_loss cuối kỳ.")
    else:
        print("✗ Val loss cuối kỳ vẫn cao hơn/bằng EXP-3 — cân nhắc tăng thêm regularization ở EXP-5/6.")

gap_to_target = 0.95 - exp4_summary["best_val_accuracy"]
print(
    f"\nKhoảng cách tới mục tiêu Accuracy >= 95%: "
    f"{gap_to_target*100:.2f} điểm % (hiện tại: {exp4_summary['best_val_accuracy']*100:.2f}%)"
)


In [ ]:
# =========================
# Ghi log vào experiments_log.csv (tổng hợp toàn bộ pipeline EXP-1..FINAL)
# =========================

log_row = pd.DataFrame([{
    "exp_name": EXP_NAME,
    "description": EXP_DESCRIPTION,
    "model": MODEL_NAME,
    "finetuned_from": CURRENT_BEST_CHECKPOINT,
    "learning_rate": LEARNING_RATE,
    "lr_stem_stage1": LR_STEM_STAGE1,
    "lr_stage2": LR_STAGE2,
    "lr_stage3": LR_STAGE3,
    "lr_stage4_head": LR_STAGE4_HEAD,
    "scheduler": "CosineAnnealingLR",
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "dropout_p": DROPOUT_P,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "best_epoch": exp4_summary["best_epoch"],
    "best_val_f1": exp4_summary["best_val_f1"],
    "best_val_roc_auc": exp4_summary["best_val_roc_auc"],
    "best_val_accuracy": exp4_summary["best_val_accuracy"],
    "best_val_loss": exp4_summary["best_val_loss"],
    "total_training_time_sec": exp4_summary["total_training_time_sec"],
    "checkpoint": exp4_summary["checkpoint"],
    "is_new_best": is_new_best
}])

if os.path.exists(EXPERIMENTS_LOG_PATH):
    existing_log = pd.read_csv(EXPERIMENTS_LOG_PATH)
    existing_log = existing_log[existing_log["exp_name"] != EXP_NAME]
    full_log = pd.concat([existing_log, log_row], ignore_index=True)
else:
    full_log = log_row

full_log.to_csv(EXPERIMENTS_LOG_PATH, index=False)

print("Saved:", EXPERIMENTS_LOG_PATH)
display(full_log)


In [ ]:
# =========================
# So sánh trực quan: BASELINE vs EXP-1 vs EXP-2 vs EXP-3 vs EXP-4
# =========================

baseline_history_path = os.path.join(
    EXPERIMENT_DIR, "histories", "convnext_tiny_history.csv"
)

exp1_history_path = os.path.join(
    FT_HISTORY_DIR, "convnext_tiny_exp1_lr_low_history.csv"
)

exp2_history_path = os.path.join(
    FT_HISTORY_DIR, "convnext_tiny_exp2_unfreeze_discriminative_lr_history.csv"
)

exp3_history_path = os.path.join(
    FT_HISTORY_DIR, "convnext_tiny_exp3_cosine_scheduler_history.csv"
)

baseline_history = pd.read_csv(baseline_history_path)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metrics_to_plot = [
    ("val_accuracy", "Validation Accuracy (mục tiêu >= 0.95)"),
    ("val_f1", "Validation F1"),
    ("val_roc_auc", "Validation ROC-AUC"),
    ("val_loss", "Validation Loss"),
]

series_to_load = [
    ("BASELINE", baseline_history_path),
    ("EXP-1 (LR=1e-5)", exp1_history_path),
    ("EXP-2 (discriminative LR)", exp2_history_path),
    ("EXP-3 (cosine scheduler)", exp3_history_path),
]

for ax, (col, title) in zip(axes, metrics_to_plot):

    for label, path in series_to_load:
        if os.path.exists(path):
            hist = pd.read_csv(path)
            if col in hist.columns:
                ax.plot(hist["epoch"], hist[col], marker="o", label=label)

    ax.plot(exp4_history["epoch"], exp4_history[col], marker="o", linewidth=2, label="EXP-4 (loss+reg)")

    if col == "val_accuracy":
        ax.axhline(0.95, color="red", linestyle="--", linewidth=1, label="Mục tiêu 95%")

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(col)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
print("EXP-4 hoàn tất.")
print()
print("Checkpoint của EXP-4         :", exp4_summary["checkpoint"])
print("Checkpoint TỐT NHẤT hiện tại (dùng cho EXP-5):", best_tracker["checkpoint"])
print()
print(f"Accuracy tốt nhất hiện tại: {best_tracker['best_val_accuracy']*100:.2f}%  (mục tiêu: 95%)")
print()
print("-> Notebook EXP-5 (Augmentation nhẹ) chỉ cần đọc lại best_tracker.json")
print("   (cell 'Best tracker' ở trên) để tự động fine-tune tiếp từ checkpoint")
print("   tốt nhất tính đến thời điểm này.")
print()
print("Lưu ý: nếu accuracy vẫn xa mục tiêu 95% sau EXP-4, khoảng cách lớn có thể")
print("đến từ giới hạn của chính việc chỉ fine-tune với LR rất thấp trên checkpoint")
print("đã gần hội tụ — cân nhắc augmentation mạnh hơn ở EXP-5, và nếu vẫn không đủ,")
print("có thể cần train lại với LR khởi điểm cao hơn (gần baseline) thay vì chỉ")
print("fine-tune nhẹ từ checkpoint hiện có.")
